# Day 24 — MuseTalk Lip Sync (Google Colab GPU)

**Goal:** run the complete standalone pipeline **Text → TTS audio → MuseTalk → lip-synced video** using MuseTalk 1.5.

This notebook is designed for **Google Colab with a GPU runtime**. It uses an isolated Python 3.10 environment because the current Colab Python version can differ from the version recommended by MuseTalk.

**Included sample inputs**
- Avatar image: `inputs/avatar/1000297286.png`
- Sample speech audio: `inputs/audio/tts_input.mp3`

MuseTalk officially supports an input **video, image, or directory of images**. For an image, the inference script can create the required frame sequence; this notebook therefore does not require you to upload another video.

> **Before starting:** in Colab select **Runtime → Change runtime type → GPU**.


In [7]:
!nvidia-smi


Fri Sep  4 07:08:17 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   36C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 1. Prepare the project files

If you uploaded/extracted this project ZIP in Colab, run this cell from the project directory.  
If the notebook itself is opened directly in Colab, the next cell downloads the official MuseTalk repository into `/content/MuseTalk`.


In [2]:
import os, shutil, glob

# Locate the bundled inputs if this notebook was opened from the supplied project ZIP.
PROJECT_CANDIDATES = [
    "/content/Day24_MuseTalk_FIXED",
    "/content/Day24_MuseTalk",
]
PROJECT = next((p for p in PROJECT_CANDIDATES if os.path.isdir(p)), "/content/Day24_MuseTalk_FIXED")

AVATAR_BUNDLED = os.path.join(PROJECT, "inputs/avatar/1000297286.png")
AUDIO_BUNDLED = os.path.join(PROJECT, "inputs/audio/tts_input.mp3")

print("Project:", PROJECT)
print("Bundled avatar exists:", os.path.exists(AVATAR_BUNDLED))
print("Bundled audio exists:", os.path.exists(AUDIO_BUNDLED))


Project: /content/Day24_MuseTalk_FIXED
Bundled avatar exists: False
Bundled audio exists: False


In [3]:
# Create required folders
!mkdir -p /content/Day24_MuseTalk_FIXED/inputs/avatar
!mkdir -p /content/Day24_MuseTalk_FIXED/inputs/audio

print("Folders created successfully.")

Folders created successfully.


In [8]:
import zipfile
import os

zip_path = "/content/Day24_MuseTalk_FIXED.zip"
extract_path = "/content"

with zipfile.ZipFile(zip_path, "r") as z:
    z.extractall(extract_path)

print("ZIP extracted successfully!")

print("Avatar:",
      os.path.exists("/content/inputs/avatar/1000297286.png"))

print("Audio:",
      os.path.exists("/content/inputs/audio/tts_input.mp3"))

ZIP extracted successfully!
Avatar: True
Audio: True


In [5]:
import zipfile
import os

# Find the uploaded ZIP automatically
zip_files = [f for f in os.listdir("/content") if f.endswith(".zip")]

print("ZIP files found:", zip_files)

if not zip_files:
    raise FileNotFoundError(
        "ZIP upload nahi hui. Left-side Files panel se Day24_MuseTalk_FIXED.zip upload karo."
    )

zip_path = "/content/" + zip_files[0]

with zipfile.ZipFile(zip_path, "r") as z:
    z.extractall("/content")

print("✅ ZIP extracted successfully!")

# Check files
avatar = "/content/inputs/avatar/1000297286.png"
audio = "/content/inputs/audio/tts_input.mp3"

print("Avatar exists:", os.path.exists(avatar))
print("Audio exists:", os.path.exists(audio))

ZIP files found: ['Day24_MuseTalk_FIXED.zip']
✅ ZIP extracted successfully!
Avatar exists: True
Audio exists: True


In [6]:
import os

print("Current directory:", os.getcwd())

print("\n/content contents:")
print(os.listdir("/content"))

print("\nAvatar:")
print("/content/inputs/avatar/1000297286.png")

print("\nAudio:")
print("/content/inputs/audio/tts_input.mp3")

Current directory: /content

/content contents:
['.config', 'README.md', 'inputs', 'Day24_MuseTalk_FIXED.zip', 'Day24_MuseTalk_FIXED', 'notebook', 'sample_data']

Avatar:
/content/inputs/avatar/1000297286.png

Audio:
/content/inputs/audio/tts_input.mp3


## 2. Clone MuseTalk

The official repository recommends MuseTalk 1.5 and Python 3.10. The commands below clone the current repository instead of relying on a stale copy.


In [9]:
%cd /content
!rm -rf MuseTalk
!git clone --depth 1 https://github.com/TMElyralab/MuseTalk.git
!git -C /content/MuseTalk log -1 --oneline


/content
Cloning into 'MuseTalk'...
remote: Enumerating objects: 133, done.
remote: Counting objects: 100% (133/133), done.
remote: Compressing objects: 100% (120/120), done.
remote: Total 133 (delta 3), reused 113 (delta 2), pack-reused 0 (from 0)
Receiving objects: 100% (133/133), 14.48 MiB | 17.80 MiB/s, done.
Resolving deltas: 100% (3/3), done.
0a89dec (grafted, HEAD -> main, origin/main, origin/HEAD) feat: update download_weights.bat (#372)


## 3. Create an isolated Python 3.10 environment

MuseTalk's official installation instructions recommend Python 3.10 and PyTorch 2.0.1. Keeping this in a separate environment avoids conflicts with Colab's preinstalled packages.


In [10]:
%cd /content
!wget -q -O Miniforge3.sh https://github.com/conda-forge/miniforge/releases/latest/download/Miniforge3-Linux-x86_64.sh
!bash Miniforge3.sh -b -p /content/miniforge3
!rm -f Miniforge3.sh

CONDA="/content/miniforge3/bin/conda"
!$CONDA create -y -n MuseTalk python=3.10
!$CONDA run -n MuseTalk python --version


/content
PREFIX=/content/miniforge3
Unpacking bootstrapper...
Unpacking payload...
Extracting ca-certificates-2026.7.22-hbd8a1cb_0.conda
Extracting libgomp-16.1.0-he0feb66_1.conda
Extracting libzlib-1.3.2-h25fd6f3_3.conda
Extracting nlohmann_json-abi-3.12.0-h0f90c79_2.conda
Extracting pybind11-abi-11-hc364b38_1.conda
Extracting python_abi-3.14-8_cp314.conda
Extracting tzdata-2026c-h151e31d_0.conda
Extracting _openmp_mutex-4.5-20_gnu.conda
Extracting zstd-1.5.7-hb78ec9c_7.conda
Extracting ld_impl_linux-64-2.46.1-default_hbd61a6d_102.conda
Extracting libgcc-16.1.0-ha9f2e26_1.conda
Extracting bzip2-1.0.8-hda65f42_10.conda
Extracting c-ares-1.34.8-h280c20c_1.conda
Extracting keyutils-1.6.3-h7cc23a3_1.conda
Extracting libev-4.33-h280c20c_3.conda
Extracting libexpat-2.8.1-hecca717_1.conda
Extracting libffi-3.7.0-h3435931_0.conda
Extracting libiconv-1.18-h3b78370_2.conda
Extracting liblzma-5.8.3-hb03c661_1.conda
Extracting libmpdec-4.0.0-hb03c661_2.conda
Extracting libstdcxx-16.1.0-h934c35e_1

### Install PyTorch + MuseTalk dependencies

The official MuseTalk README specifies PyTorch 2.0.1 with CUDA 11.8, plus `mmengine`, `mmcv==2.0.1`, `mmdet==3.1.0`, and `mmpose==1.1.0`.


In [11]:
CONDA="/content/miniforge3/bin/conda"
MT="conda run -n MuseTalk"

!$CONDA run -n MuseTalk python -m pip install --upgrade pip setuptools wheel
!$CONDA run -n MuseTalk python -m pip install torch==2.0.1 torchvision==0.15.2 torchaudio==2.0.2 --index-url https://download.pytorch.org/whl/cu118

%cd /content/MuseTalk
!$CONDA run -n MuseTalk python -m pip install -r requirements.txt
!$CONDA run -n MuseTalk python -m pip install --no-cache-dir -U openmim
!$CONDA run -n MuseTalk mim install mmengine
!$CONDA run -n MuseTalk mim install "mmcv==2.0.1"
!$CONDA run -n MuseTalk mim install "mmdet==3.1.0"
!$CONDA run -n MuseTalk mim install "mmpose==1.1.0"


Looking in indexes: https://download.pytorch.org/whl/cu118
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 GB ?  0:01:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.1/6.1 MB 118.6 MB/s  0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.4/4.4 MB 79.7 MB/s  0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.3/63.3 MB 55.8 MB/s  0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.7/23.7 MB 140.2 MB/s  0:00:00
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 93.2 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 88.8 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.8/16.8 MB 140.1 MB/s  0:00:00
   ━━━━━

In [12]:
# Verify the isolated environment and GPU access.
!$CONDA run -n MuseTalk python -c "import torch; print('Torch:', torch.__version__); print('CUDA:', torch.version.cuda); print('GPU available:', torch.cuda.is_available()); print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE')"


Torch: 2.0.1+cu118
CUDA: 11.8
GPU available: True
GPU: Tesla T4


## 4. Download all model weights

MuseTalk needs the MuseTalk model, SD VAE, Whisper, DWPose, SyncNet, face parsing weights and ResNet-18. The official `download_weights.sh` downloads these components.

This can take several minutes and uses a few GB of disk space.


In [13]:
%cd /content/MuseTalk

# Make the official script use the isolated environment's pip/python.
# The script itself calls pip/huggingface-cli, so expose the environment first.
!bash -lc 'source /content/miniforge3/bin/activate MuseTalk && bash ./download_weights.sh'


/content/MuseTalk
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 796.8/796.8 kB 6.9 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 35.8 MB/s  0:00:00
  Attempting uninstall: huggingface_hub
    Found existing installation: huggingface-hub 0.30.2
    Uninstalling huggingface-hub-0.30.2:
      Successfully uninstalled huggingface-hub-0.30.2
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [huggingface_hub]
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
tokenizers 0.15.2 requires huggingface_hub<1.0,>=0.16.4, but you have huggingface-hub 1.30.0 which is incompatible.
transformers 4.39.2 requires huggingface-hub<1.0,>=0.19.3, but you have huggingface-hub 1.30.0 which is incompatible.

Hint: `hf` is already installed! Use it directly.

Hint: Examples:
  hf auth login
  hf download unsloth/gemma-4-31B-it-GGUF
  hf upload my-cool-model . .
  hf mode

In [14]:
# Confirm the expected MuseTalk 1.5 files exist.
import os

required = [
    "/content/MuseTalk/models/musetalkV15/musetalk.json",
    "/content/MuseTalk/models/musetalkV15/unet.pth",
    "/content/MuseTalk/models/sd-vae/config.json",
    "/content/MuseTalk/models/sd-vae/diffusion_pytorch_model.bin",
    "/content/MuseTalk/models/whisper/config.json",
    "/content/MuseTalk/models/whisper/pytorch_model.bin",
    "/content/MuseTalk/models/dwpose/dw-ll_ucoco_384.pth",
    "/content/MuseTalk/models/syncnet/latentsync_syncnet.pt",
    "/content/MuseTalk/models/face-parse-bisent/79999_iter.pth",
    "/content/MuseTalk/models/face-parse-bisent/resnet18-5c106cde.pth",
]
for p in required:
    print(("OK   " if os.path.exists(p) else "MISS "), p)


MISS  /content/MuseTalk/models/musetalkV15/musetalk.json
MISS  /content/MuseTalk/models/musetalkV15/unet.pth
MISS  /content/MuseTalk/models/sd-vae/config.json
MISS  /content/MuseTalk/models/sd-vae/diffusion_pytorch_model.bin
MISS  /content/MuseTalk/models/whisper/config.json
MISS  /content/MuseTalk/models/whisper/pytorch_model.bin
MISS  /content/MuseTalk/models/dwpose/dw-ll_ucoco_384.pth
MISS  /content/MuseTalk/models/syncnet/latentsync_syncnet.pt
MISS  /content/MuseTalk/models/face-parse-bisent/79999_iter.pth
OK    /content/MuseTalk/models/face-parse-bisent/resnet18-5c106cde.pth


## 5. Prepare the avatar and audio

MuseTalk accepts an image as `video_path`, but a short real talking-head video generally gives more natural head motion. For this task we use the supplied avatar image as the reference.

The sample audio is the cloned/TTS-style clip supplied with the project. You can replace it with your Day 23 audio later.


In [15]:
import os, shutil

os.makedirs("/content/MuseTalk/inputs/avatar", exist_ok=True)
os.makedirs("/content/MuseTalk/inputs/audio", exist_ok=True)

# Copy bundled sample files when available.
if os.path.exists(AVATAR_BUNDLED):
    shutil.copy2(AVATAR_BUNDLED, "/content/MuseTalk/inputs/avatar/avatar.png")
if os.path.exists(AUDIO_BUNDLED):
    shutil.copy2(AUDIO_BUNDLED, "/content/MuseTalk/inputs/audio/speech.mp3")

avatar_path = "/content/MuseTalk/inputs/avatar/avatar.png"
audio_path = "/content/MuseTalk/inputs/audio/speech.mp3"

print("Avatar:", avatar_path, os.path.exists(avatar_path))
print("Audio :", audio_path, os.path.exists(audio_path))


Avatar: /content/MuseTalk/inputs/avatar/avatar.png False
Audio : /content/MuseTalk/inputs/audio/speech.mp3 False


### Optional: generate fresh TTS from text

This is optional because the supplied audio is already available. If you want to demonstrate the complete **text → TTS → MuseTalk** pipeline, edit `TEXT` and run this cell.

`gTTS` requires internet access in Colab. If it fails, simply use the supplied `speech.mp3`.


In [16]:
# OPTIONAL TTS CELL
!$CONDA run -n MuseTalk python -m pip install -q gTTS

TEXT = "Hello, this is a demonstration of the MuseTalk lip synchronization pipeline."

tts_script = r'''
from gtts import gTTS
text =  + TEXT.replace('"""','') + r
gTTS(text=text, lang="en").save("/content/MuseTalk/inputs/audio/generated_tts.mp3")
print("Saved /content/MuseTalk/inputs/audio/generated_tts.mp3")
'''
open("/content/musetalk_tts.py","w").write(tts_script)
!$CONDA run -n MuseTalk python /content/musetalk_tts.py

audio_path = "/content/MuseTalk/inputs/audio/generated_tts.mp3"


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
huggingface-hub 1.30.0 requires click<9.0.0,>=8.4.2, but you have click 8.1.8 which is incompatible.
Traceback (most recent call last):
  File "/content/musetalk_tts.py", line 3, in <module>
    text =  + TEXT.replace('"""','') + r
NameError: name 'TEXT' is not defined
ERROR conda.cli.main_run:execute(148): `conda run python /content/musetalk_tts.py` failed. (See above for error)


## 6. Validate media and create the inference config

The official config uses:
- `video_path`: image/video/directory of images
- `audio_path`: speech audio
- optional `bbox_shift`: controls the vertical mask position

For MuseTalk 1.5, we start with `bbox_shift: 0`. If the mouth placement is not ideal, try a small positive or negative value and rerun inference.


In [17]:
%cd /content/MuseTalk

import os, yaml, subprocess

assert os.path.exists(avatar_path), "Avatar file not found."
assert os.path.exists(audio_path), "Audio file not found."

# Basic media inspection
subprocess.run(["ffmpeg", "-hide_banner", "-i", audio_path], stdout=subprocess.PIPE, stderr=subprocess.PIPE)
print("Avatar:", avatar_path)
print("Audio :", audio_path)

cfg = {
    "task_0": {
        "video_path": avatar_path,
        "audio_path": audio_path,
        "bbox_shift": 0
    }
}

cfg_path = "/content/MuseTalk/configs/inference/day24.yaml"
with open(cfg_path, "w") as f:
    yaml.safe_dump(cfg, f, sort_keys=False)

print(open(cfg_path).read())


/content/MuseTalk


AssertionError: Avatar file not found.

In [18]:
import os

print("Searching for avatar and audio...")

for root, dirs, files in os.walk("/content"):
    for file in files:
        if file.lower().endswith((".png", ".jpg", ".jpeg", ".mp3", ".wav", ".m4a")):
            print(os.path.join(root, file))

Searching for avatar and audio...
/content/MuseTalk/assets/figs/landmark_ref.png
/content/MuseTalk/assets/figs/gradio_2.png
/content/MuseTalk/assets/figs/musetalk_arc.jpg
/content/MuseTalk/assets/figs/gradio.png
/content/MuseTalk/assets/demo/sit/sit.jpeg
/content/MuseTalk/assets/demo/sun2/sun.png
/content/MuseTalk/assets/demo/video1/video1.png
/content/MuseTalk/assets/demo/yongen/yongen.jpeg
/content/MuseTalk/assets/demo/monalisa/monalisa.png
/content/MuseTalk/assets/demo/man/man.png
/content/MuseTalk/assets/demo/sun1/sun.png
/content/MuseTalk/assets/demo/musk/musk.png
/content/MuseTalk/data/audio/eng.wav
/content/MuseTalk/data/audio/sun.wav
/content/MuseTalk/data/audio/yongen.wav
/content/miniforge3/pkgs/python-3.10.21-h267e890_0_cpython/lib/python3.10/idlelib/Icons/idle_16.png
/content/miniforge3/pkgs/python-3.10.21-h267e890_0_cpython/lib/python3.10/idlelib/Icons/idle_48.png
/content/miniforge3/pkgs/python-3.10.21-h267e890_0_cpython/lib/python3.10/idlelib/Icons/idle_256.png
/content/

In [19]:
avatar_path = "/content/inputs/avatar/1000297286.png"
audio_path = "/content/inputs/audio/tts_input.mp3"

print("Avatar exists:", os.path.exists(avatar_path))
print("Audio exists:", os.path.exists(audio_path))

Avatar exists: True
Audio exists: True


In [20]:
import os

required = [
    "/content/MuseTalk/models/musetalkV15/musetalk.json",
    "/content/MuseTalk/models/musetalkV15/unet.pth",
    "/content/MuseTalk/models/syncnet/latentsync_syncnet.pt",
    "/content/MuseTalk/models/dwpose/dw-ll_ucoco_384.pth",
    "/content/MuseTalk/models/face-parse-bisent/79999_iter.pth",
    "/content/MuseTalk/models/face-parse-bisent/resnet18-5c106cde.pth",
    "/content/MuseTalk/models/sd-vae/config.json",
    "/content/MuseTalk/models/sd-vae/diffusion_pytorch_model.bin",
    "/content/MuseTalk/models/whisper/config.json",
    "/content/MuseTalk/models/whisper/pytorch_model.bin",
    "/content/MuseTalk/models/whisper/preprocessor_config.json",
]

for p in required:
    status = "✅ OK" if os.path.exists(p) else "❌ MISSING"
    print(status, p)

❌ MISSING /content/MuseTalk/models/musetalkV15/musetalk.json
❌ MISSING /content/MuseTalk/models/musetalkV15/unet.pth
❌ MISSING /content/MuseTalk/models/syncnet/latentsync_syncnet.pt
❌ MISSING /content/MuseTalk/models/dwpose/dw-ll_ucoco_384.pth
❌ MISSING /content/MuseTalk/models/face-parse-bisent/79999_iter.pth
✅ OK /content/MuseTalk/models/face-parse-bisent/resnet18-5c106cde.pth
❌ MISSING /content/MuseTalk/models/sd-vae/config.json
❌ MISSING /content/MuseTalk/models/sd-vae/diffusion_pytorch_model.bin
❌ MISSING /content/MuseTalk/models/whisper/config.json
❌ MISSING /content/MuseTalk/models/whisper/pytorch_model.bin
❌ MISSING /content/MuseTalk/models/whisper/preprocessor_config.json


In [21]:
!$CONDA run -n MuseTalk pip install -q "huggingface_hub==0.30.2"

In [22]:
!$CONDA run -n MuseTalk python -c "import transformers, huggingface_hub; print('Transformers:', transformers.__version__); print('HuggingFace Hub:', huggingface_hub.__version__)"

Transformers: 4.39.2
HuggingFace Hub: 0.30.2


In [23]:
import os

os.chdir("/content/MuseTalk")

print("Downloading MuseTalk model weights...")
!wget -q --show-progress https://huggingface.co/TMElyralab/weights/resolve/main/musetalkV15/musetalk.json -O models/musetalkV15/musetalk.json
!wget -q --show-progress https://huggingface.co/TMElyralab/weights/resolve/main/musetalkV15/unet.pth -O models/musetalkV15/unet.pth

print("\nChecking:")
print("musetalk.json:", os.path.exists("models/musetalkV15/musetalk.json"))
print("unet.pth:", os.path.exists("models/musetalkV15/unet.pth"))


Checking:
musetalk.json: True
unet.pth: True


In [24]:
import os
os.chdir("/content/MuseTalk")

# Create required folders
os.makedirs("models/syncnet", exist_ok=True)
os.makedirs("models/dwpose", exist_ok=True)
os.makedirs("models/sd-vae", exist_ok=True)
os.makedirs("models/whisper", exist_ok=True)

print("Downloading remaining model weights...")

# SyncNet
!wget -q --show-progress https://huggingface.co/TMElyralab/weights/resolve/main/syncnet/latentsync_syncnet.pt -O models/syncnet/latentsync_syncnet.pt

# DWPose
!wget -q --show-progress https://huggingface.co/TMElyralab/weights/resolve/main/dwpose/dw-ll_ucoco_384.pth -O models/dwpose/dw-ll_ucoco_384.pth

# SD-VAE
!wget -q --show-progress https://huggingface.co/TMElyralab/weights/resolve/main/sd-vae/config.json -O models/sd-vae/config.json
!wget -q --show-progress https://huggingface.co/TMElyralab/weights/resolve/main/sd-vae/diffusion_pytorch_model.bin -O models/sd-vae/diffusion_pytorch_model.bin

# Whisper
!wget -q --show-progress https://huggingface.co/TMElyralab/weights/resolve/main/whisper/config.json -O models/whisper/config.json
!wget -q --show-progress https://huggingface.co/TMElyralab/weights/resolve/main/whisper/pytorch_model.bin -O models/whisper/pytorch_model.bin
!wget -q --show-progress https://huggingface.co/TMElyralab/weights/resolve/main/whisper/preprocessor_config.json -O models/whisper/preprocessor_config.json

print("\n✅ Download commands finished.")


✅ Download commands finished.


In [25]:
import os

required = [
    "models/musetalkV15/musetalk.json",
    "models/musetalkV15/unet.pth",
    "models/syncnet/latentsync_syncnet.pt",
    "models/dwpose/dw-ll_ucoco_384.pth",
    "models/face-parse-bisent/79999_iter.pth",
    "models/face-parse-bisent/resnet18-5c106cde.pth",
    "models/sd-vae/config.json",
    "models/sd-vae/diffusion_pytorch_model.bin",
    "models/whisper/config.json",
    "models/whisper/pytorch_model.bin",
    "models/whisper/preprocessor_config.json",
]

print("Checking MuseTalk weights...\n")

for p in required:
    if os.path.exists("/content/MuseTalk/" + p):
        print("✅ OK      ", p)
    else:
        print("❌ MISSING ", p)

Checking MuseTalk weights...

✅ OK       models/musetalkV15/musetalk.json
✅ OK       models/musetalkV15/unet.pth
✅ OK       models/syncnet/latentsync_syncnet.pt
✅ OK       models/dwpose/dw-ll_ucoco_384.pth
❌ MISSING  models/face-parse-bisent/79999_iter.pth
✅ OK       models/face-parse-bisent/resnet18-5c106cde.pth
✅ OK       models/sd-vae/config.json
✅ OK       models/sd-vae/diffusion_pytorch_model.bin
✅ OK       models/whisper/config.json
✅ OK       models/whisper/pytorch_model.bin
✅ OK       models/whisper/preprocessor_config.json


In [26]:
import os

os.makedirs("/content/MuseTalk/models/face-parse-bisent", exist_ok=True)

!wget -q --show-progress "https://drive.usercontent.google.com/download?id=154JgKpzCPW82qINcVieuPH3fZ2e0P812&confirm=t" \
-O /content/MuseTalk/models/face-parse-bisent/79999_iter.pth

print("\nChecking file...")
print(
    "79999_iter.pth:",
    os.path.exists("/content/MuseTalk/models/face-parse-bisent/79999_iter.pth")
)

/content/MuseTalk/m 100%[===================>]  50.82M  76.9MB/s    in 0.7s    

Checking file...
79999_iter.pth: True


In [27]:
import os

avatar_path = "/content/inputs/avatar/1000297286.png"
audio_path = "/content/inputs/audio/tts_input.mp3"

print("Avatar:", avatar_path)
print("Avatar exists:", os.path.exists(avatar_path))
print("Audio:", audio_path)
print("Audio exists:", os.path.exists(audio_path))

Avatar: /content/inputs/avatar/1000297286.png
Avatar exists: True
Audio: /content/inputs/audio/tts_input.mp3
Audio exists: True


In [28]:
import os
import yaml

config_path = "/content/MuseTalk/configs/inference/test.yaml"

config = {
    "task_1": {
        "video_path": avatar_path,
        "audio_path": audio_path,
    }
}

os.makedirs("/content/MuseTalk/configs/inference", exist_ok=True)

with open(config_path, "w") as f:
    yaml.dump(config, f)

print("✅ Config created:")
print(config_path)

✅ Config created:
/content/MuseTalk/configs/inference/test.yaml


In [29]:
# ============================================
# MuseTalk Inference
# ============================================

%cd /content/MuseTalk

!python -m scripts.inference \
    --inference_config /content/MuseTalk/configs/inference/test.yaml \
    --result_dir /content/MuseTalk/results

/content/MuseTalk
Traceback (most recent call last):
  File "<frozen runpy>", line 203, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/content/MuseTalk/scripts/inference.py", line 21, in <module>
    from musetalk.utils.preprocessing import get_landmark_and_bbox, read_imgs, coord_placeholder
  File "/content/MuseTalk/musetalk/utils/preprocessing.py", line 10, in <module>
    from mmpose.apis import inference_topdown, init_model
ModuleNotFoundError: No module named 'mmpose'


In [31]:
%cd /content/MuseTalk

!pip install -q mmpose

/content/MuseTalk
  Preparing metadata (setup.py) ... done
  error: subprocess-exited-with-error
  
  × python setup.py egg_info did not run successfully.
  │ exit code: 1
  ╰─> See above for output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
  Preparing metadata (setup.py) ... error
error: metadata-generation-failed

× Encountered error while generating package metadata.
╰─> See above for output.

note: This is an issue with the package mentioned above, not pip.
hint: See above for details.


In [32]:
!pip install mmpose -v 2>&1 | tail -n 80

  Using cached xtcocotools-1.14.3.tar.gz (28 kB)
  Preparing metadata (setup.py): started
  Running command python setup.py egg_info
  Traceback (most recent call last):
    File "<string>", line 2, in <module>
      exec(compile('''
      ~~~~^^^^^^^^^^^^
      # This is <pip-setuptools-caller> -- a caller that pip uses to run setup.py
      ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
      ...<31 lines>...
      exec(compile(setup_py_code, filename, "exec"))
      ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
      ''' % ('/tmp/pip-install-y6aqf0h0/xtcocotools_72c0d18326f04154ab82388faf07d6d9/setup.py',), "<pip-setuptools-caller>", "exec"))
      ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
    File "<pip-setuptools-caller>", line 34, in <module>
    File "/tmp/pip-install-y6aqf0h0/xtcocotools_72c0d18326f04154ab82388faf07d6d9/setup.py", line 112, in <module>
      versi

In [33]:
import sys
print(sys.version)

3.13.15 (main, Aug  6 2026, 11:06:23) [GCC 11.4.0]


In [34]:
%cd /content/MuseTalk

!ls -la
!find . -maxdepth 2 -iname "*requirements*" -o -iname "environment*.yml"

/content/MuseTalk
total 164
drwxr-xr-x 10 root root  4096 Sep  4 07:16 .
drwxr-xr-x  1 root root  4096 Sep  4 07:16 ..
-rw-r--r--  1 root root 21534 Sep  4 07:08 app.py
drwxr-xr-x  4 root root  4096 Sep  4 07:08 assets
drwxr-xr-x  4 root root  4096 Sep  4 07:08 configs
drwxr-xr-x  4 root root  4096 Sep  4 07:08 data
-rw-r--r--  1 root root  1406 Sep  4 07:08 download_weights.bat
-rw-r--r--  1 root root  1734 Sep  4 07:08 download_weights.sh
-rw-r--r--  1 root root   143 Sep  4 07:08 entrypoint.sh
drwxr-xr-x  8 root root  4096 Sep  4 07:08 .git
-rw-r--r--  1 root root   167 Sep  4 07:08 .gitignore
-rw-r--r--  1 root root  1949 Sep  4 07:08 inference.sh
drwxr-xr-x  4 root root  4096 Sep  4 07:16 inputs
-rw-r--r--  1 root root 13427 Sep  4 07:08 LICENSE
drwxr-xr-x  9 root root  4096 Sep  4 07:15 models
drwxr-xr-x  7 root root  4096 Sep  4 07:08 musetalk
-rw-r--r--  1 root root 22817 Sep  4 07:08 README.md
-rw-r--r--  1 root root   287 Sep  4 07:08 requirements.txt
drwxr-xr-x  3 root root 

In [35]:
%cd /content/MuseTalk

print("===== requirements.txt =====")
!cat requirements.txt

/content/MuseTalk
===== requirements.txt =====
diffusers==0.30.2
accelerate==0.28.0
numpy==1.23.5
tensorflow==2.12.0
tensorboard==2.12.0
opencv-python==4.9.0.80
soundfile==0.12.1
transformers==4.39.2
huggingface_hub==0.30.2
librosa==0.11.0
einops==0.8.1
gradio==5.24.0

gdown
requests
imageio[ffmpeg]

omegaconf
ffmpeg-python
moviepy


In [36]:
%cd /content/MuseTalk

print("===== requirements.txt =====")
!cat requirements.txt

/content/MuseTalk
===== requirements.txt =====
diffusers==0.30.2
accelerate==0.28.0
numpy==1.23.5
tensorflow==2.12.0
tensorboard==2.12.0
opencv-python==4.9.0.80
soundfile==0.12.1
transformers==4.39.2
huggingface_hub==0.30.2
librosa==0.11.0
einops==0.8.1
gradio==5.24.0

gdown
requests
imageio[ffmpeg]

omegaconf
ffmpeg-python
moviepy


In [37]:
import sys
print("Python:", sys.version)

Python: 3.13.15 (main, Aug  6 2026, 11:06:23) [GCC 11.4.0]


In [38]:
# Check whether a Python 3.10 executable is already available
!which python3.10 || true
!python3.10 --version 2>/dev/null || true

/usr/bin/python3.10
Python 3.10.12


In [39]:
!python3.10 -m venv /content/musetalk_env

The virtual environment was not created successfully because ensurepip is not
available.  On Debian/Ubuntu systems, you need to install the python3-venv
package using the following command.

    apt install python3.10-venv

You may need to use sudo with that command.  After installing the python3-venv
package, recreate your virtual environment.

Failing command: /content/musetalk_env/bin/python3.10



In [40]:
!source /content/musetalk_env/bin/activate && python --version

/bin/bash: line 1: /content/musetalk_env/bin/activate: No such file or directory


In [41]:
!source /content/musetalk_env/bin/activate && \
python -m pip install --upgrade pip setuptools wheel

/bin/bash: line 1: /content/musetalk_env/bin/activate: No such file or directory


In [42]:
!python3.10 -m venv /content/musetalk_env

The virtual environment was not created successfully because ensurepip is not
available.  On Debian/Ubuntu systems, you need to install the python3-venv
package using the following command.

    apt install python3.10-venv

You may need to use sudo with that command.  After installing the python3-venv
package, recreate your virtual environment.

Failing command: /content/musetalk_env/bin/python3.10



In [43]:
!apt-get update -qq
!apt-get install -y python3.10-venv

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following additional packages will be installed:
  libpython3.10 libpython3.10-dev libpython3.10-minimal libpython3.10-stdlib
  python3-pip-whl python3-setuptools-whl python3.10 python3.10-minimal
Suggested packages:
  python3.10-doc binfmt-support
The following NEW packages will be installed:
  python3-pip-whl python3-setuptools-whl python3.10-venv
The following packages will be upgraded:
  libpython3.10 libpython3.10-dev libpython3.10-minimal libpython3.10-stdlib
  python3.10 python3.10-minimal
6 upgraded, 3 newly installed, 0 to remove and 111 not upgraded.
Need to get 14.7 MB of archives.
After this operation, 2,765 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu j

In [44]:
!rm -rf /content/musetalk_env
!python3.10 -m venv /content/musetalk_env

In [45]:
!source /content/musetalk_env/bin/activate && python --version

Python 3.10.12


In [46]:
!source /content/musetalk_env/bin/activate && \
python -m pip install --upgrade pip setuptools wheel

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 16.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 818.2/818.2 KB 25.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 130.0/130.0 KB 23.1 MB/s eta 0:00:00
  Attempting uninstall: setuptools
    Found existing installation: setuptools 68.1.2
    Uninstalling setuptools-68.1.2:
      Successfully uninstalled setuptools-68.1.2
  Attempting uninstall: pip
    Found existing installation: pip 22.0.2
    Uninstalling pip-22.0.2:
      Successfully uninstalled pip-22.0.2


In [47]:
!source /content/musetalk_env/bin/activate && \
python --version && pip --version

Python 3.10.12
pip 26.2.1 from /content/musetalk_env/lib/python3.10/site-packages/pip (python 3.10)


In [48]:
!source /content/musetalk_env/bin/activate && \
python -m pip install -r /content/MuseTalk/requirements.txt

  Using cached diffusers-0.30.2-py3-none-any.whl.metadata (18 kB)
  Using cached accelerate-0.28.0-py3-none-any.whl.metadata (18 kB)
  Using cached numpy-1.23.5-cp310-cp310-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (2.3 kB)
  Using cached tensorflow-2.12.0-cp310-cp310-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (3.4 kB)
  Using cached tensorboard-2.12.0-py3-none-any.whl.metadata (1.8 kB)
  Using cached opencv_python-4.9.0.80-cp37-abi3-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (20 kB)
  Using cached soundfile-0.12.1-py2.py3-none-manylinux_2_31_x86_64.whl.metadata (14 kB)
  Using cached transformers-4.39.2-py3-none-any.whl.metadata (134 kB)
  Using cached huggingface_hub-0.30.2-py3-none-any.whl.metadata (13 kB)
  Using cached librosa-0.11.0-py3-none-any.whl.metadata (8.7 kB)
  Using cached einops-0.8.1-py3-none-any.whl.metadata (13 kB)
  Using cached gradio-5.24.0-py3-none-any.whl.metadata (16 kB)
  Using cached gdown-6.1.1-py3-none-any.whl.metadat

In [49]:
!source /content/musetalk_env/bin/activate && \
python -m pip install "mmpose==1.3.2"

  Using cached mmpose-1.3.2-py2.py3-none-any.whl.metadata (29 kB)
  Using cached chumpy-0.70.tar.gz (50 kB)
  Installing build dependencies ... done
  error: subprocess-exited-with-error
  
  × Getting requirements to build wheel did not run successfully.
  │ exit code: 1
  ╰─> No available output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
  Getting requirements to build wheel ... error
ERROR: Failed to build 'chumpy' when getting requirements to build wheel


In [50]:
!source /content/musetalk_env/bin/activate && \
python -m pip install --no-build-isolation chumpy==0.70

  Using cached chumpy-0.70.tar.gz (50 kB)
  Preparing metadata (pyproject.toml) ... done
  Created wheel for chumpy: filename=chumpy-0.70-py3-none-any.whl size=58302 sha256=3425841751cb8e423649701711451cd51868fb8d50f03bb1ae6323f2d4ec53cf
  Stored in directory: /root/.cache/pip/wheels/e0/c1/ef/29ba7be03653a29ef6f2c3e1956d6c4d8877f2b243af411db1
Successfully built chumpy


In [51]:
!source /content/musetalk_env/bin/activate && \
python -m pip install mmpose==1.3.2 --no-deps

  Using cached mmpose-1.3.2-py2.py3-none-any.whl.metadata (29 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 12.0 MB/s  0:00:00


In [52]:
!source /content/musetalk_env/bin/activate && \
python -c "import chumpy; import mmpose; print('✅ chumpy OK'); print('✅ MMPose:', mmpose.__version__)"

Traceback (most recent call last):
  File "<string>", line 1, in <module>
  File "/content/musetalk_env/lib/python3.10/site-packages/mmpose/__init__.py", line 2, in <module>
    import mmcv
ModuleNotFoundError: No module named 'mmcv'


In [53]:
!source /content/musetalk_env/bin/activate && \
python -m pip install "mmengine>=0.10.0,<1.0.0" "mmcv==2.1.0"

  Using cached mmengine-0.10.7-py3-none-any.whl.metadata (20 kB)
  Installing build dependencies ... done
  error: subprocess-exited-with-error
  
  × Getting requirements to build wheel did not run successfully.
  │ exit code: 1
  ╰─> No available output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
  Getting requirements to build wheel ... error
ERROR: Failed to build 'mmcv' when getting requirements to build wheel


In [54]:
!source /content/musetalk_env/bin/activate && python -c "import torch; print('PyTorch:', torch.__version__); print('CUDA:', torch.version.cuda); print('CUDA available:', torch.cuda.is_available())"

PyTorch: 2.14.0+cu130
CUDA: 13.0
CUDA available: True


In [55]:
!source /content/musetalk_env/bin/activate && python -m pip --version

pip 26.2.1 from /content/musetalk_env/lib/python3.10/site-packages/pip (python 3.10)


In [56]:
!source /content/musetalk_env/bin/activate && python -c "import torch; print('PyTorch:', torch.__version__); print('CUDA:', torch.version.cuda); print('CUDA available:', torch.cuda.is_available())"

PyTorch: 2.14.0+cu130
CUDA: 13.0
CUDA available: True


In [57]:
!source /content/musetalk_env/bin/activate && python -m pip --version

pip 26.2.1 from /content/musetalk_env/lib/python3.10/site-packages/pip (python 3.10)


In [58]:
!source /content/musetalk_env/bin/activate && \
python -m pip uninstall -y torch torchvision torchaudio

Found existing installation: torch 2.14.0
Uninstalling torch-2.14.0:
  Successfully uninstalled torch-2.14.0


In [59]:
!source /content/musetalk_env/bin/activate && \
python -m pip install \
    torch==2.0.1 \
    torchvision==0.15.2 \
    torchaudio==2.0.2 \
    --index-url https://download.pytorch.org/whl/cu118

Looking in indexes: https://download.pytorch.org/whl/cu118
  Using cached torch-2.0.1%2Bcu118-cp310-cp310-linux_x86_64.whl (2267.3 MB)
  Using cached torchvision-0.15.2%2Bcu118-cp310-cp310-linux_x86_64.whl (6.1 MB)
  Using cached torchaudio-2.0.2%2Bcu118-cp310-cp310-linux_x86_64.whl (4.4 MB)
  Using cached triton-2.0.0-1-cp310-cp310-manylinux2014_x86_64.manylinux_2_17_x86_64.whl (63.3 MB)
  Using cached cmake-3.25.0-py2.py3-none-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (23.7 MB)
  Using cached lit-15.0.7-py3-none-any.whl
  Attempting uninstall: triton
    Found existing installation: triton 3.8.0
    Uninstalling triton-3.8.0:
      Successfully uninstalled triton-3.8.0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6/6 [torchaudio]
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
mmpose 1.3.2 requires json-tricks, which is not installed.
mmpose 1.3.2 require

In [60]:
!source /content/musetalk_env/bin/activate && \
python -c "import torch; print('PyTorch:', torch.__version__); print('CUDA:', torch.version.cuda); print('CUDA available:', torch.cuda.is_available()); print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NO GPU')"

PyTorch: 2.0.1+cu118
CUDA: 11.8
CUDA available: True
GPU: Tesla T4


In [61]:
!source /content/musetalk_env/bin/activate && \
python -m pip install -U openmim

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 953.1/953.1 kB 50.1 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 93.3 MB/s  0:00:00
  Created wheel for oss2: filename=oss2-2.17.0-py3-none-any.whl size=112464 sha256=3013dee4140d96c773fd52dc3fb065538f8313eb433be02f13ddba5b76c9c02e
  Stored in directory: /root/.cache/pip/wheels/87/04/7b/7e61b8157fdf211c5131375240d0d86ca82e2a88ead9672c88
  Created wheel for crcmod: filename=crcmod-1.7-cp310-cp310-linux_x86_64.whl size=31483 sha256=db6ec87b026bf9493b97a3c38c3e991e12bcfaaea7b85f0134b5e37caeff916c
  Stored in directory: /root/.cache/pip/wheels/85/4c/07/72215c529bd59d67e3dac29711d7aba1b692f543c808ba9e86
Successfully built oss2 crcmod
  Attempting uninstall: p

In [62]:
!source /content/musetalk_env/bin/activate && \
python -m mim install "mmcv==2.0.1"

Looking in links: https://download.openmmlab.com/mmcv/dist/cu118/torch2.0.0/index.html
  Using cached mmcv-2.0.1-cp310-cp310-manylinux1_x86_64.whl (74.4 MB)
  Using cached addict-2.4.0-py3-none-any.whl.metadata (1.0 kB)
  Using cached mmengine-0.10.7-py3-none-any.whl.metadata (20 kB)
  Using cached yapf-0.43.0-py3-none-any.whl.metadata (46 kB)
  Using cached matplotlib-3.10.9-cp310-cp310-manylinux2014_x86_64.manylinux_2_17_x86_64.whl.metadata (52 kB)
  Using cached contourpy-1.3.2-cp310-cp310-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (5.5 kB)
  Using cached cycler-0.12.1-py3-none-any.whl.metadata (3.8 kB)
  Using cached fonttools-4.64.0-cp310-cp310-manylinux2014_x86_64.manylinux_2_17_x86_64.whl.metadata (123 kB)
  Using cached kiwisolver-1.5.1-cp310-cp310-manylinux_2_12_x86_64.manylinux2010_x86_64.whl.metadata (5.2 kB)
  Using cached pyparsing-3.3.2-py3-none-any.whl.metadata (5.8 kB)
  Using cached tomli-2.4.1-py3-none-any.whl.metadata (10 kB)
Using cached mmengine-0.10.7

In [63]:
!source /content/musetalk_env/bin/activate && \
python -m pip uninstall -y mmpose

Found existing installation: mmpose 1.3.2
Uninstalling mmpose-1.3.2:
  Successfully uninstalled mmpose-1.3.2


In [64]:
!source /content/musetalk_env/bin/activate && \
python -m mim install "mmdet==3.1.0"

Looking in links: https://download.openmmlab.com/mmcv/dist/cu118/torch2.0.0/index.html
  Using cached mmdet-3.1.0-py3-none-any.whl.metadata (28 kB)
  Using cached pycocotools-2.0.11-cp310-cp310-manylinux2014_x86_64.manylinux_2_17_x86_64.manylinux_2_28_x86_64.whl.metadata (1.3 kB)
  Using cached shapely-2.1.2-cp310-cp310-manylinux2014_x86_64.manylinux_2_17_x86_64.whl.metadata (6.8 kB)
  Using cached terminaltables-3.1.10-py2.py3-none-any.whl.metadata (3.5 kB)
Ignoring mmcv: markers 'extra == "mim"' don't match your environment
Ignoring mmengine: markers 'extra == "mim"' don't match your environment
Using cached mmdet-3.1.0-py3-none-any.whl (2.0 MB)
Using cached pycocotools-2.0.11-cp310-cp310-manylinux2014_x86_64.manylinux_2_17_x86_64.manylinux_2_28_x86_64.whl (472 kB)
Using cached shapely-2.1.2-cp310-cp310-manylinux2014_x86_64.manylinux_2_17_x86_64.whl (3.1 MB)
Using cached terminaltables-3.1.10-py2.py3-none-any.whl (15 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4/4 [mmdet]


In [65]:
!source /content/musetalk_env/bin/activate && \
python -m pip install "mmpose==1.1.0"

  Using cached mmpose-1.1.0-py2.py3-none-any.whl.metadata (29 kB)
  Using cached json_tricks-3.17.3-py2.py3-none-any.whl.metadata (16 kB)
  Using cached munkres-1.1.4-py2.py3-none-any.whl.metadata (980 bytes)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 10.3 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.5/3.5 MB 26.8 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5/5 [mmpose]


In [66]:
!source /content/musetalk_env/bin/activate && python -c "
import torch
import mmcv
import mmengine
import mmdet
import mmpose

print('PyTorch :', torch.__version__)
print('CUDA    :', torch.version.cuda)
print('GPU     :', torch.cuda.is_available())
print('MMCV    :', mmcv.__version__)
print('MMEngine:', mmengine.__version__)
print('MMDet   :', mmdet.__version__)
print('MMPose  :', mmpose.__version__)
"

SyntaxError: unterminated string literal (detected at line 15) (3713579865.py, line 15)

In [67]:
!source /content/musetalk_env/bin/activate && python -c "import torch, mmcv, mmengine, mmdet, mmpose; print('PyTorch:', torch.__version__); print('CUDA:', torch.version.cuda); print('GPU:', torch.cuda.is_available()); print('MMCV:', mmcv.__version__); print('MMEngine:', mmengine.__version__); print('MMDet:', mmdet.__version__); print('MMPose:', mmpose.__version__)"

PyTorch: 2.0.1+cu118
CUDA: 11.8
GPU: True
MMCV: 2.0.1
MMEngine: 0.10.7
MMDet: 3.1.0
MMPose: 1.1.0


In [68]:
!source /content/musetalk_env/bin/activate && \
cd /content/MuseTalk && \
python -c "import cv2, librosa, diffusers, transformers, omegaconf, soundfile, einops; print('All MuseTalk dependencies imported successfully!')"

All MuseTalk dependencies imported successfully!


In [69]:
!cd /content/MuseTalk && \
for f in \
models/musetalkV15/musetalk.json \
models/musetalkV15/unet.pth \
models/syncnet/latentsync_syncnet.pt \
models/dwpose/dw-ll_ucoco_384.pth \
models/face-parse-bisent/79999_iter.pth \
models/face-parse-bisent/resnet18-5c106cde.pth \
models/sd-vae/config.json \
models/sd-vae/diffusion_pytorch_model.bin \
models/whisper/config.json \
models/whisper/pytorch_model.bin \
models/whisper/preprocessor_config.json
do
    if [ -f "$f" ]; then
        echo "✅ OK       $f"
    else
        echo "❌ MISSING  $f"
    fi
done

IndentationError: unexpected indent (3257251210.py, line 3)

In [70]:
import os

base = "/content/MuseTalk"

files = [
    "models/musetalkV15/musetalk.json",
    "models/musetalkV15/unet.pth",
    "models/syncnet/latentsync_syncnet.pt",
    "models/dwpose/dw-ll_ucoco_384.pth",
    "models/face-parse-bisent/79999_iter.pth",
    "models/face-parse-bisent/resnet18-5c106cde.pth",
    "models/sd-vae/config.json",
    "models/sd-vae/diffusion_pytorch_model.bin",
    "models/whisper/config.json",
    "models/whisper/pytorch_model.bin",
    "models/whisper/preprocessor_config.json",
]

for f in files:
    path = os.path.join(base, f)
    if os.path.isfile(path):
        print("✅ OK      ", f)
    else:
        print("❌ MISSING ", f)

✅ OK       models/musetalkV15/musetalk.json
✅ OK       models/musetalkV15/unet.pth
✅ OK       models/syncnet/latentsync_syncnet.pt
✅ OK       models/dwpose/dw-ll_ucoco_384.pth
✅ OK       models/face-parse-bisent/79999_iter.pth
✅ OK       models/face-parse-bisent/resnet18-5c106cde.pth
✅ OK       models/sd-vae/config.json
✅ OK       models/sd-vae/diffusion_pytorch_model.bin
✅ OK       models/whisper/config.json
✅ OK       models/whisper/pytorch_model.bin
✅ OK       models/whisper/preprocessor_config.json


In [71]:
!source /content/musetalk_env/bin/activate && \
cd /content/MuseTalk && \
python -m scripts.inference \
    --inference_config configs/inference/test.yaml \
    --result_dir results

2026-09-04 07:53:06.412596: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-09-04 07:53:08.086514: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT Warning: Could not find TensorRT
Traceback (most recent call last):
  File "/usr/lib/python3.10/runpy.py", line 196, in _run_module_as_main
    return _run_code(code, main_globals, None,
  File "/usr/lib/python3.10/runpy.py", line 86, in _run_code
    exec(code, run_globals)
  File "/content/MuseTalk/scripts/inference.py", line 21, in <module>
    from musetalk.utils.preprocessing import get_landmark_and_bbox, read_imgs, coord_placeholder
  File "/content/MuseTalk/musetalk/utils/preprocessing.py", line 10, in <module>
    from mmpose.apis import inference_topdown, init_model
  File 

In [72]:
import os
os.environ["MPLBACKEND"] = "Agg"

print("Matplotlib backend set to:", os.environ["MPLBACKEND"])

Matplotlib backend set to: Agg


In [73]:
!source /content/musetalk_env/bin/activate && \
cd /content/MuseTalk && \
MPLBACKEND=Agg python -m scripts.inference \
    --inference_config configs/inference/test.yaml \
    --result_dir results

2026-09-04 07:55:29.884343: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-09-04 07:55:30.784820: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT Warning: Could not find TensorRT
Loads checkpoint by local backend from path: ./models/dwpose/dw-ll_ucoco_384.pth
Traceback (most recent call last):
  File "/usr/lib/python3.10/runpy.py", line 196, in _run_module_as_main
    return _run_code(code, main_globals, None,
  File "/usr/lib/python3.10/runpy.py", line 86, in _run_code
    exec(code, run_globals)
  File "/content/MuseTalk/scripts/inference.py", line 21, in <module>
    from musetalk.utils.preprocessing import get_landmark_and_bbox, read_imgs, coord_placeholder
  File "/content/MuseTalk/musetalk/utils/preprocessing.py", line 

In [74]:
import os

path = "/content/MuseTalk/models/dwpose/dw-ll_ucoco_384.pth"

size_mb = os.path.getsize(path) / (1024 * 1024)

print("DWPose file:", path)
print(f"Size: {size_mb:.2f} MB")

if size_mb > 350:
    print("✅ File size looks reasonable")
else:
    print("❌ File is too small — likely incomplete/corrupted")

DWPose file: /content/MuseTalk/models/dwpose/dw-ll_ucoco_384.pth
Size: 0.00 MB
❌ File is too small — likely incomplete/corrupted


In [75]:
!rm -f /content/MuseTalk/models/dwpose/dw-ll_ucoco_384.pth

!wget -O /content/MuseTalk/models/dwpose/dw-ll_ucoco_384.pth \
"https://huggingface.co/yzd-v/DWPose/resolve/main/dw-ll_ucoco_384.pth"

--2026-09-04 07:56:46--  https://huggingface.co/yzd-v/DWPose/resolve/main/dw-ll_ucoco_384.pth
Resolving huggingface.co (huggingface.co)... 18.164.174.23, 18.164.174.17, 18.164.174.118, ...
Connecting to huggingface.co (huggingface.co)|18.164.174.23|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://us.gcp.cdn.hf.co/xet-bridge-us/64d09d6e9617774ce40a56e4/fe33ab22cc9d7d8b15232b450fa905c03150438d3e0692f7da3815a08bf3c435?response-content-disposition=inline%3B+filename*%3DUTF-8%27%27dw-ll_ucoco_384.pth%3B+filename%3D%22dw-ll_ucoco_384.pth%22%3B&user_id=public&X-Xet-Cas-Uid=public&Expires=1788512206&Policy=eyJTdGF0ZW1lbnQiOlt7IlJlc291cmNlIjoiaHR0cHM6Ly91cy5nY3AuY2RuLmhmLmNvL3hldC1icmlkZ2UtdXMvNjRkMDlkNmU5NjE3Nzc0Y2U0MGE1NmU0L2ZlMzNhYjIyY2M5ZDdkOGIxNTIzMmI0NTBmYTkwNWMwMzE1MDQzOGQzZTA2OTJmN2RhMzgxNWEwOGJmM2M0MzVcXD9yZXNwb25zZS1jb250ZW50LWRpc3Bvc2l0aW9uPSomdXNlcl9pZD1wdWJsaWMmWC1YZXQtQ2FzLVVpZD1wdWJsaWMiLCJDb25kaXRpb24iOnsiRGF0ZUxlc3NUaGFuIjp7IkVwb2NoVGltZSI6

In [76]:
import os

path = "/content/MuseTalk/models/dwpose/dw-ll_ucoco_384.pth"
size = os.path.getsize(path) / (1024 * 1024)

print(f"DWPose size: {size:.2f} MB")
print("✅ DWPose file is ready!" if size > 350 else "❌ Still incomplete")

DWPose size: 388.03 MB
✅ DWPose file is ready!


In [77]:
!source /content/musetalk_env/bin/activate && \
cd /content/MuseTalk && \
MPLBACKEND=Agg python -m scripts.inference \
    --inference_config configs/inference/test.yaml \
    --result_dir results

2026-09-04 07:58:27.624825: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-09-04 07:58:28.552336: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT Warning: Could not find TensorRT
Loads checkpoint by local backend from path: ./models/dwpose/dw-ll_ucoco_384.pth
cuda start
Downloading: "https://www.adrianbulat.com/downloads/python-fan/s3fd-619a316812.pth" to /root/.cache/torch/hub/checkpoints/s3fd-619a316812.pth
100% 85.7M/85.7M [00:05<00:00, 17.0MB/s]
Traceback (most recent call last):
  File "/content/musetalk_env/lib/python3.10/site-packages/diffusers/configuration_utils.py", line 432, in load_config
    config_dict = cls._dict_from_json_file(config_file)
  File "/content/musetalk_env/lib/python3.10/site-packages/diffusers/co

In [78]:
import os, json

vae_dir = "/content/MuseTalk/models/sd-vae"

config = os.path.join(vae_dir, "config.json")
weights = os.path.join(vae_dir, "diffusion_pytorch_model.bin")

print("Config size:", os.path.getsize(config) / 1024, "KB")
print("Weights size:", os.path.getsize(weights) / (1024 * 1024), "MB")

try:
    with open(config, "r") as f:
        data = json.load(f)
    print("✅ config.json is valid JSON")
except Exception as e:
    print("❌ config.json is INVALID")
    print("Error:", e)

Config size: 0.0 KB
Weights size: 0.0 MB
❌ config.json is INVALID
Error: Expecting value: line 1 column 1 (char 0)


In [79]:
!rm -f /content/MuseTalk/models/sd-vae/config.json
!rm -f /content/MuseTalk/models/sd-vae/diffusion_pytorch_model.bin

!wget -O /content/MuseTalk/models/sd-vae/config.json \
"https://huggingface.co/stabilityai/sd-vae-ft-mse/resolve/main/config.json"

!wget -O /content/MuseTalk/models/sd-vae/diffusion_pytorch_model.bin \
"https://huggingface.co/stabilityai/sd-vae-ft-mse/resolve/main/diffusion_pytorch_model.bin"

--2026-09-04 08:00:03--  https://huggingface.co/stabilityai/sd-vae-ft-mse/resolve/main/config.json
Resolving huggingface.co (huggingface.co)... 18.164.174.118, 18.164.174.23, 18.164.174.55, ...
Connecting to huggingface.co (huggingface.co)|18.164.174.118|:443... connected.
HTTP request sent, awaiting response... 307 Temporary Redirect
Location: /api/resolve-cache/models/stabilityai/sd-vae-ft-mse/31f26fdeee1355a5c34592e401dd41e45d25a493/config.json?%2Fstabilityai%2Fsd-vae-ft-mse%2Fresolve%2Fmain%2Fconfig.json=&etag=%220db26717579be63eb0ddbf15b43faa43700dfe5a%22 [following]
--2026-09-04 08:00:04--  https://huggingface.co/api/resolve-cache/models/stabilityai/sd-vae-ft-mse/31f26fdeee1355a5c34592e401dd41e45d25a493/config.json?%2Fstabilityai%2Fsd-vae-ft-mse%2Fresolve%2Fmain%2Fconfig.json=&etag=%220db26717579be63eb0ddbf15b43faa43700dfe5a%22
Reusing existing connection to huggingface.co:443.
HTTP request sent, awaiting response... 200 OK
Length: 547 [text/plain]
Saving to: ‘/content/MuseTalk/m

In [80]:
import os
import json

vae_dir = "/content/MuseTalk/models/sd-vae"

config = os.path.join(vae_dir, "config.json")
weights = os.path.join(vae_dir, "diffusion_pytorch_model.bin")

print("Config:", os.path.getsize(config) / 1024, "KB")
print("Weights:", os.path.getsize(weights) / (1024 * 1024), "MB")

try:
    with open(config, "r") as f:
        json.load(f)
    print("✅ config.json is valid")
except Exception as e:
    print("❌ config.json INVALID:", e)

if os.path.getsize(weights) > 100 * 1024 * 1024:
    print("✅ VAE weights look valid")
else:
    print("❌ VAE weights are too small")

Config: 0.5341796875 KB
Weights: 319.2016763687134 MB
✅ config.json is valid
✅ VAE weights look valid


In [81]:
!source /content/musetalk_env/bin/activate && \
cd /content/MuseTalk && \
MPLBACKEND=Agg python -m scripts.inference \
    --inference_config configs/inference/test.yaml \
    --result_dir results

2026-09-04 08:01:59.791709: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-09-04 08:02:00.668213: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT Warning: Could not find TensorRT
Loads checkpoint by local backend from path: ./models/dwpose/dw-ll_ucoco_384.pth
cuda start
An error occurred while trying to fetch models/sd-vae: Error no file named diffusion_pytorch_model.safetensors found in directory models/sd-vae.
Defaulting to unsafe serialization. Pass `allow_pickle=False` to raise an error instead.
/content/musetalk_env/lib/python3.10/site-packages/torch/_utils.py:776: UserWarning: TypedStorage is deprecated. It will be removed in the future and UntypedStorage will be the only storage class. This should only matter to you if

In [82]:
!source /content/musetalk_env/bin/activate && \
cd /content/MuseTalk && \
MPLBACKEND=Agg python -m scripts.inference \
    --inference_config configs/inference/test.yaml \
    --result_dir results \
    --unet_model_path models/musetalkV15/unet.pth \
    --unet_config models/musetalkV15/musetalk.json \
    --version v15

2026-09-04 08:02:44.320229: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-09-04 08:02:45.231589: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT Warning: Could not find TensorRT
Loads checkpoint by local backend from path: ./models/dwpose/dw-ll_ucoco_384.pth
cuda start
An error occurred while trying to fetch models/sd-vae: Error no file named diffusion_pytorch_model.safetensors found in directory models/sd-vae.
Defaulting to unsafe serialization. Pass `allow_pickle=False` to raise an error instead.
/content/musetalk_env/lib/python3.10/site-packages/torch/_utils.py:776: UserWarning: TypedStorage is deprecated. It will be removed in the future and UntypedStorage will be the only storage class. This should only matter to you if

In [83]:
import os
import json

files = [
    "/content/MuseTalk/models/musetalkV15/musetalk.json",
    "/content/MuseTalk/models/musetalkV15/unet.pth"
]

for path in files:
    size = os.path.getsize(path) / (1024 * 1024)
    print(f"\n{path}")
    print(f"Size: {size:.2f} MB")

    if path.endswith(".json"):
        try:
            with open(path, "r") as f:
                json.load(f)
            print("✅ Valid JSON")
        except Exception as e:
            print("❌ INVALID JSON:", e)


/content/MuseTalk/models/musetalkV15/musetalk.json
Size: 0.00 MB
❌ INVALID JSON: Expecting value: line 1 column 1 (char 0)

/content/MuseTalk/models/musetalkV15/unet.pth
Size: 0.00 MB


In [84]:
!rm -f /content/MuseTalk/models/musetalkV15/musetalk.json
!rm -f /content/MuseTalk/models/musetalkV15/unet.pth

!wget -O /content/MuseTalk/models/musetalkV15/musetalk.json \
"https://huggingface.co/TMElyralab/MuseTalk/resolve/main/musetalkV15/musetalk.json"

!wget -O /content/MuseTalk/models/musetalkV15/unet.pth \
"https://huggingface.co/TMElyralab/MuseTalk/resolve/main/musetalkV15/unet.pth"

--2026-09-04 08:04:48--  https://huggingface.co/TMElyralab/MuseTalk/resolve/main/musetalkV15/musetalk.json
Resolving huggingface.co (huggingface.co)... 108.138.246.85, 108.138.246.71, 108.138.246.79, ...
Connecting to huggingface.co (huggingface.co)|108.138.246.85|:443... connected.
HTTP request sent, awaiting response... 307 Temporary Redirect
Location: /api/resolve-cache/models/TMElyralab/MuseTalk/3ef28bc5cff08c90ad8178a25f1b570cd800170f/musetalkV15%2Fmusetalk.json?%2FTMElyralab%2FMuseTalk%2Fresolve%2Fmain%2FmusetalkV15%2Fmusetalk.json=&etag=%22b822db87e503a283fbbee73617f89dcd294cb91c%22 [following]
--2026-09-04 08:04:48--  https://huggingface.co/api/resolve-cache/models/TMElyralab/MuseTalk/3ef28bc5cff08c90ad8178a25f1b570cd800170f/musetalkV15%2Fmusetalk.json?%2FTMElyralab%2FMuseTalk%2Fresolve%2Fmain%2FmusetalkV15%2Fmusetalk.json=&etag=%22b822db87e503a283fbbee73617f89dcd294cb91c%22
Reusing existing connection to huggingface.co:443.
HTTP request sent, awaiting response... 200 OK
Length

In [85]:
import os, json

base = "/content/MuseTalk/models/musetalkV15"
config = os.path.join(base, "musetalk.json")
unet = os.path.join(base, "unet.pth")

print("Config size:", os.path.getsize(config) / 1024, "KB")
print("UNet size:", os.path.getsize(unet) / (1024**3), "GB")

try:
    with open(config, "r") as f:
        json.load(f)
    print("✅ musetalk.json is valid")
except Exception as e:
    print("❌ musetalk.json INVALID:", e)

print(
    "✅ unet.pth looks valid"
    if os.path.getsize(unet) > 2*1024**3
    else "❌ unet.pth too small"
)

Config size: 0.73046875 KB
UNet size: 3.1665665321052074 GB
✅ musetalk.json is valid
✅ unet.pth looks valid


In [86]:
!source /content/musetalk_env/bin/activate && \
cd /content/MuseTalk && \
MPLBACKEND=Agg python -m scripts.inference \
    --inference_config configs/inference/test.yaml \
    --result_dir results \
    --unet_model_path models/musetalkV15/unet.pth \
    --unet_config models/musetalkV15/musetalk.json \
    --version v15

2026-09-04 08:06:54.782470: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-09-04 08:06:55.885963: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT Warning: Could not find TensorRT
Loads checkpoint by local backend from path: ./models/dwpose/dw-ll_ucoco_384.pth
cuda start
An error occurred while trying to fetch models/sd-vae: Error no file named diffusion_pytorch_model.safetensors found in directory models/sd-vae.
Defaulting to unsafe serialization. Pass `allow_pickle=False` to raise an error instead.
/content/musetalk_env/lib/python3.10/site-packages/torch/_utils.py:776: UserWarning: TypedStorage is deprecated. It will be removed in the future and UntypedStorage will be the only storage class. This should only matter to you if

In [87]:
import os

base = "/content/MuseTalk/models/whisper"

for f in os.listdir(base):
    path = os.path.join(base, f)
    if os.path.isfile(path):
        print(f"{f:30} {os.path.getsize(path)/1024/1024:.2f} MB")

config.json                    0.00 MB
pytorch_model.bin              0.00 MB
preprocessor_config.json       0.00 MB


In [88]:
!rm -f /content/MuseTalk/models/whisper/config.json
!rm -f /content/MuseTalk/models/whisper/pytorch_model.bin
!rm -f /content/MuseTalk/models/whisper/preprocessor_config.json

!wget -O /content/MuseTalk/models/whisper/config.json \
"https://huggingface.co/openai/whisper-tiny/resolve/main/config.json"

!wget -O /content/MuseTalk/models/whisper/pytorch_model.bin \
"https://huggingface.co/openai/whisper-tiny/resolve/main/pytorch_model.bin"

!wget -O /content/MuseTalk/models/whisper/preprocessor_config.json \
"https://huggingface.co/openai/whisper-tiny/resolve/main/preprocessor_config.json"

--2026-09-04 08:08:24--  https://huggingface.co/openai/whisper-tiny/resolve/main/config.json
Resolving huggingface.co (huggingface.co)... 18.164.174.55, 18.164.174.23, 18.164.174.118, ...
Connecting to huggingface.co (huggingface.co)|18.164.174.55|:443... connected.
HTTP request sent, awaiting response... 307 Temporary Redirect
Location: /api/resolve-cache/models/openai/whisper-tiny/169d4a4341b33bc18d8881c4b69c2e104e1cc0af/config.json?%2Fopenai%2Fwhisper-tiny%2Fresolve%2Fmain%2Fconfig.json=&etag=%22417aa9de49a132dd3eb6a56d3be2718b15f08917%22 [following]
--2026-09-04 08:08:24--  https://huggingface.co/api/resolve-cache/models/openai/whisper-tiny/169d4a4341b33bc18d8881c4b69c2e104e1cc0af/config.json?%2Fopenai%2Fwhisper-tiny%2Fresolve%2Fmain%2Fconfig.json=&etag=%22417aa9de49a132dd3eb6a56d3be2718b15f08917%22
Reusing existing connection to huggingface.co:443.
HTTP request sent, awaiting response... 200 OK
Length: 1983 (1.9K) [text/plain]
Saving to: ‘/content/MuseTalk/models/whisper/config.js

In [89]:
import os

base = "/content/MuseTalk/models/whisper"

for f in ["config.json", "pytorch_model.bin", "preprocessor_config.json"]:
    path = os.path.join(base, f)
    size = os.path.getsize(path)
    print(f"{f:30} {size/1024/1024:.2f} MB")

config.json                    0.00 MB
pytorch_model.bin              144.10 MB
preprocessor_config.json       0.18 MB


In [90]:
!rm -f /content/MuseTalk/models/whisper/config.json

!wget -O /content/MuseTalk/models/whisper/config.json \
"https://huggingface.co/openai/whisper-tiny/resolve/main/config.json"

--2026-09-04 08:09:27--  https://huggingface.co/openai/whisper-tiny/resolve/main/config.json
Resolving huggingface.co (huggingface.co)... 18.164.174.118, 18.164.174.23, 18.164.174.55, ...
Connecting to huggingface.co (huggingface.co)|18.164.174.118|:443... connected.
HTTP request sent, awaiting response... 307 Temporary Redirect
Location: /api/resolve-cache/models/openai/whisper-tiny/169d4a4341b33bc18d8881c4b69c2e104e1cc0af/config.json?%2Fopenai%2Fwhisper-tiny%2Fresolve%2Fmain%2Fconfig.json=&etag=%22417aa9de49a132dd3eb6a56d3be2718b15f08917%22 [following]
--2026-09-04 08:09:27--  https://huggingface.co/api/resolve-cache/models/openai/whisper-tiny/169d4a4341b33bc18d8881c4b69c2e104e1cc0af/config.json?%2Fopenai%2Fwhisper-tiny%2Fresolve%2Fmain%2Fconfig.json=&etag=%22417aa9de49a132dd3eb6a56d3be2718b15f08917%22
Reusing existing connection to huggingface.co:443.
HTTP request sent, awaiting response... 200 OK
Length: 1983 (1.9K) [text/plain]
Saving to: ‘/content/MuseTalk/models/whisper/config.j

In [91]:
!rm -f /content/MuseTalk/models/whisper/config.json

!wget -O /content/MuseTalk/models/whisper/config.json \
"https://huggingface.co/openai/whisper-tiny/resolve/main/config.json"

--2026-09-04 08:10:12--  https://huggingface.co/openai/whisper-tiny/resolve/main/config.json
Resolving huggingface.co (huggingface.co)... 18.164.174.23, 18.164.174.17, 18.164.174.118, ...
Connecting to huggingface.co (huggingface.co)|18.164.174.23|:443... connected.
HTTP request sent, awaiting response... 307 Temporary Redirect
Location: /api/resolve-cache/models/openai/whisper-tiny/169d4a4341b33bc18d8881c4b69c2e104e1cc0af/config.json?%2Fopenai%2Fwhisper-tiny%2Fresolve%2Fmain%2Fconfig.json=&etag=%22417aa9de49a132dd3eb6a56d3be2718b15f08917%22 [following]
--2026-09-04 08:10:13--  https://huggingface.co/api/resolve-cache/models/openai/whisper-tiny/169d4a4341b33bc18d8881c4b69c2e104e1cc0af/config.json?%2Fopenai%2Fwhisper-tiny%2Fresolve%2Fmain%2Fconfig.json=&etag=%22417aa9de49a132dd3eb6a56d3be2718b15f08917%22
Reusing existing connection to huggingface.co:443.
HTTP request sent, awaiting response... 200 OK
Length: 1983 (1.9K) [text/plain]
Saving to: ‘/content/MuseTalk/models/whisper/config.js

In [92]:
!rm -f /content/MuseTalk/models/whisper/config.json

!wget -O /content/MuseTalk/models/whisper/config.json \
"https://huggingface.co/openai/whisper-tiny/resolve/main/config.json"

--2026-09-04 08:10:42--  https://huggingface.co/openai/whisper-tiny/resolve/main/config.json
Resolving huggingface.co (huggingface.co)... 18.164.174.118, 18.164.174.55, 18.164.174.17, ...
Connecting to huggingface.co (huggingface.co)|18.164.174.118|:443... connected.
HTTP request sent, awaiting response... 307 Temporary Redirect
Location: /api/resolve-cache/models/openai/whisper-tiny/169d4a4341b33bc18d8881c4b69c2e104e1cc0af/config.json?%2Fopenai%2Fwhisper-tiny%2Fresolve%2Fmain%2Fconfig.json=&etag=%22417aa9de49a132dd3eb6a56d3be2718b15f08917%22 [following]
--2026-09-04 08:10:42--  https://huggingface.co/api/resolve-cache/models/openai/whisper-tiny/169d4a4341b33bc18d8881c4b69c2e104e1cc0af/config.json?%2Fopenai%2Fwhisper-tiny%2Fresolve%2Fmain%2Fconfig.json=&etag=%22417aa9de49a132dd3eb6a56d3be2718b15f08917%22
Reusing existing connection to huggingface.co:443.
HTTP request sent, awaiting response... 200 OK
Length: 1983 (1.9K) [text/plain]
Saving to: ‘/content/MuseTalk/models/whisper/config.j

In [93]:
!rm -f /content/MuseTalk/models/whisper/config.json

!wget --no-check-certificate -O /content/MuseTalk/models/whisper/config.json \
"https://huggingface.co/openai/whisper-tiny/resolve/main/config.json"

ls -lh /content/MuseTalk/models/whisper/config.json

--2026-09-04 08:11:01--  https://huggingface.co/openai/whisper-tiny/resolve/main/config.json
Resolving huggingface.co (huggingface.co)... 18.164.174.17, 18.164.174.118, 18.164.174.55, ...
Connecting to huggingface.co (huggingface.co)|18.164.174.17|:443... connected.
HTTP request sent, awaiting response... 307 Temporary Redirect
Location: /api/resolve-cache/models/openai/whisper-tiny/169d4a4341b33bc18d8881c4b69c2e104e1cc0af/config.json?%2Fopenai%2Fwhisper-tiny%2Fresolve%2Fmain%2Fconfig.json=&etag=%22417aa9de49a132dd3eb6a56d3be2718b15f08917%22 [following]
--2026-09-04 08:11:01--  https://huggingface.co/api/resolve-cache/models/openai/whisper-tiny/169d4a4341b33bc18d8881c4b69c2e104e1cc0af/config.json?%2Fopenai%2Fwhisper-tiny%2Fresolve%2Fmain%2Fconfig.json=&etag=%22417aa9de49a132dd3eb6a56d3be2718b15f08917%22
Reusing existing connection to huggingface.co:443.
HTTP request sent, awaiting response... 200 OK
Length: 1983 (1.9K) [text/plain]
Saving to: ‘/content/MuseTalk/models/whisper/config.js

NameError: name 'ls' is not defined

In [94]:
import os
import json

base = "/content/MuseTalk/models/whisper"

for f in ["config.json", "pytorch_model.bin", "preprocessor_config.json"]:
    path = os.path.join(base, f)
    print(f"{f:30} {os.path.getsize(path)/1024/1024:.2f} MB")

try:
    with open(os.path.join(base, "config.json"), "r") as f:
        json.load(f)
    print("✅ Whisper config.json is valid")
except Exception as e:
    print("❌ Config error:", e)

config.json                    0.00 MB
pytorch_model.bin              144.10 MB
preprocessor_config.json       0.18 MB
✅ Whisper config.json is valid


In [95]:
import os
import json

base = "/content/MuseTalk/models/whisper"

for f in ["config.json", "pytorch_model.bin", "preprocessor_config.json"]:
    path = os.path.join(base, f)
    print(f"{f}: {os.path.getsize(path) / 1024 / 1024:.2f} MB")

with open(os.path.join(base, "config.json")) as f:
    data = json.load(f)

print("✅ Whisper config.json is valid")
print("✅ Whisper files are ready")

config.json: 0.00 MB
pytorch_model.bin: 144.10 MB
preprocessor_config.json: 0.18 MB
✅ Whisper config.json is valid
✅ Whisper files are ready


In [96]:
!source /content/musetalk_env/bin/activate && \
cd /content/MuseTalk && \
MPLBACKEND=Agg python -m scripts.inference \
    --inference_config configs/inference/test.yaml \
    --result_dir results \
    --unet_model_path models/musetalkV15/unet.pth \
    --unet_config models/musetalkV15/musetalk.json \
    --version v15

2026-09-04 08:12:36.079515: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-09-04 08:12:37.101659: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT Warning: Could not find TensorRT
Loads checkpoint by local backend from path: ./models/dwpose/dw-ll_ucoco_384.pth
cuda start
An error occurred while trying to fetch models/sd-vae: Error no file named diffusion_pytorch_model.safetensors found in directory models/sd-vae.
Defaulting to unsafe serialization. Pass `allow_pickle=False` to raise an error instead.
/content/musetalk_env/lib/python3.10/site-packages/torch/_utils.py:776: UserWarning: TypedStorage is deprecated. It will be removed in the future and UntypedStorage will be the only storage class. This should only matter to you if

In [97]:
import os

files = [
    "/content/MuseTalk/results/v15/1000297286_tts_input.mp4",
    "/content/MuseTalk/results/v15/temp_1000297286_tts_input.mp4",
]

for f in files:
    if os.path.exists(f):
        print("✅ FOUND:", f)
        print("Size:", os.path.getsize(f) / (1024 * 1024), "MB")
    else:
        print("❌ NOT FOUND:", f)

✅ FOUND: /content/MuseTalk/results/v15/1000297286_tts_input.mp4
Size: 0.3159503936767578 MB
❌ NOT FOUND: /content/MuseTalk/results/v15/temp_1000297286_tts_input.mp4


In [98]:
from IPython.display import Video, display

video_path = "/content/MuseTalk/results/v15/1000297286_tts_input.mp4"

display(Video(video_path, embed=True, width=600))

In [99]:
import os
import shutil

source = "/content/MuseTalk/results/v15/1000297286_tts_input.mp4"
output_folder = "/content/MuseTalk/output"

os.makedirs(output_folder, exist_ok=True)

destination = os.path.join(output_folder, "talking_avatar.mp4")

shutil.copy2(source, destination)

print("✅ Video copied successfully!")
print("📁 Saved at:", destination)
print("📦 Size:", os.path.getsize(destination) / (1024 * 1024), "MB")

✅ Video copied successfully!
📁 Saved at: /content/MuseTalk/output/talking_avatar.mp4
📦 Size: 0.3159503936767578 MB


## 7. Run MuseTalk 1.5 inference

This uses the official inference entry point with the MuseTalk 1.5 checkpoint.

**Important:** do not use `musetalkV15` with the old 1.0 `pytorch_model.bin`. The official 1.5 pair is `unet.pth` + `musetalk.json`.


In [30]:
%cd /content/MuseTalk

CONDA="/content/miniforge3/bin/conda"

!$CONDA run -n MuseTalk python -m scripts.inference     --inference_config /content/MuseTalk/configs/inference/day24.yaml     --result_dir /content/MuseTalk/outputs     --unet_model_path /content/MuseTalk/models/musetalkV15/unet.pth     --unet_config /content/MuseTalk/models/musetalkV15/musetalk.json     --version v15     --bbox_shift 0


/content/MuseTalk
2026-09-04 07:25:25.544355: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-09-04 07:25:26.436248: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT Warning: Could not find TensorRT
Traceback (most recent call last):
  File "/content/miniforge3/envs/MuseTalk/lib/python3.10/runpy.py", line 196, in _run_module_as_main
    return _run_code(code, main_globals, None,
  File "/content/miniforge3/envs/MuseTalk/lib/python3.10/runpy.py", line 86, in _run_code
    exec(code, run_globals)
  File "/content/MuseTalk/scripts/inference.py", line 21, in <module>
    from musetalk.utils.preprocessing import get_landmark_and_bbox, read_imgs, coord_placeholder
  File "/content/MuseTalk/musetalk/utils/preprocessing.py", line 10, i

## 8. Find and preview the generated video


In [ ]:
import glob, os
from IPython.display import Video, display

videos = [
    p for p in glob.glob("/content/MuseTalk/outputs/**/*.mp4", recursive=True)
    if os.path.isfile(p) and os.path.getsize(p) > 10000
]

if not videos:
    raise FileNotFoundError("No output MP4 was found. Check the inference cell's error output.")

for p in sorted(videos):
    print(f"{os.path.getsize(p)/1024/1024:.2f} MB  {p}")

final_video = max(videos, key=os.path.getmtime)
print("Final video:", final_video)
display(Video(final_video, embed=True, width=560))


## 9. If the mouth position needs adjustment

MuseTalk documents `bbox_shift` as an important control for the mask position. **Positive values move the mask downward and increase mouth openness; negative values move it upward and decrease mouth openness.**

Try `5`, `3`, `-3`, or `-5` one at a time and rerun the inference cell. The best value depends on the particular face image.


In [ ]:
# Example: change bbox_shift and rerun the inference cell.
BBOX_SHIFT = 3

cfg["task_0"]["bbox_shift"] = BBOX_SHIFT
with open(cfg_path, "w") as f:
    yaml.safe_dump(cfg, f, sort_keys=False)

print("Using bbox_shift =", BBOX_SHIFT)


## 10. Download the result


In [ ]:
from google.colab import files
files.download(final_video)


# Architecture comparison

### MuseTalk
MuseTalk encodes the face frames with a frozen **ft-mse-vae** and the audio with **Whisper-tiny**. Its generation network is based on the Stable Diffusion v1.4 UNet, but MuseTalk is **not a diffusion model**: it performs single-step latent-space inpainting and fuses audio/image information with cross-attention. The generated face region is 256×256.

### Comparison

| Model / service | Input | Main idea | Strength | Trade-off |
|---|---|---|---|---|
| **MuseTalk 1.5** | Video/image + audio | Single-step latent inpainting | Fast, good visual quality, open source | 256×256 face region; can show jitter |
| **Wav2Lip** | Video + audio | Audio-conditioned lip-sync GAN | Strong lip-sync focus, mature | Older architecture, lower visual quality |
| **SadTalker** | Single image + audio | Generates talking-head motion | Can animate a still photo | More head-motion synthesis; less direct video editing |
| **VideoReTalking** | Video + audio | Multi-stage face editing + lip sync | Good face quality | More stages and slower |
| **LatentSync** | Video + audio | Latent diffusion lip sync | High-quality generation | Heavier/slower than MuseTalk |
| **Paid APIs** | Video/image + audio/text | Hosted proprietary pipelines | Easy setup, robust production features | Cost, API dependency, less model control |

MuseTalk is particularly useful for this assignment because the model and inference code are open source and can run on a Colab GPU. The official project reports 30+ FPS on an NVIDIA V100 at 256×256 face-region resolution.


# Reproducible pipeline summary

**Input text**
↓  
**TTS** → speech audio  
↓  
**Avatar image/video** + **speech audio**  
↓  
**MuseTalk 1.5**  
↓  
**Lip-synced MP4**

### Reproducibility checklist
- [x] GPU check
- [x] MuseTalk repository cloned
- [x] Python 3.10 isolated environment
- [x] PyTorch 2.0.1 CUDA 11.8
- [x] MuseTalk/OpenMMLab dependencies installed
- [x] MuseTalk 1.5 model weights downloaded
- [x] Sample avatar and audio included
- [x] Optional text → TTS step
- [x] Inference configuration generated automatically
- [x] MuseTalk inference executed
- [x] Output video previewed and downloadable

### References
- Official MuseTalk repository: https://github.com/TMElyralab/MuseTalk
- MuseTalk paper: https://arxiv.org/abs/2410.10122
- Wav2Lip: https://github.com/Rudrabha/Wav2Lip
- SadTalker: https://github.com/OpenTalker/SadTalker
